# Analyze button — crew pipeline

This notebook explains what happens when the user clicks **Analyze**: Gradio
collects inputs, the pipeline builds a five-agent CrewAI crew with MCP tools,
and a typed analysis package is produced (or a deterministic fallback).

**Source files**
- `app.py` — button wiring to the analyze callback
- `careercrew/callbacks_analyze.py` — `analyze`, `approve`, `reject`, `request_revision`
- `careercrew/pipeline.py` — `run_analysis`
- `careercrew/crew.py` — `build_crew`
- `config/agents.yaml`, `config/tasks.yaml`
- Detail doc: `docs/CODE_PATH_ANALYZE.md`


## Overview

Analyze is the core of CareerCrew. Gradio gathers the resume and JD (upload or
paste), runs input guardrails, then `pipeline.run_analysis` builds a CrewAI crew
of five agents with MCP tools.

The crew produces a typed package: profile, requirements, gaps, rewrites, and a
prep plan. If the LLM crew fails, CareerCrew still returns a
**gap-based fallback** so the demo keeps working. The result is
submitted into HITL review and rendered across the UI tabs.


## Call chain

```
Gradio Analyze
   callbacks_analyze.analyze
      resolve uploads / paste
      guardrails.validate_request
      pipeline.run_analysis
         build_crew (agents.yaml + tasks.yaml + MCP)
         crew.kickoff
         parse → Pydantic models
         OR gap-based fallback (prep.build_fallback_plan)
      review.submit_for_review
      session_store persist
   render Match / Prep tabs
```


## Agents and tools

| Piece | Location |
|-------|----------|
| Agent roles | `config/agents.yaml` |
| Task prompts | `config/tasks.yaml` |
| Crew wiring | `careercrew/crew.py` → `build_crew` |
| Shared tool logic | `careercrew/document_tools.py` |
| MCP server | `mcp_server/server.py` — `read_resume_tool`, `read_job_description_tool` |
| Contracts | `careercrew/models.py` |
| Honesty | cited resume facts only; unsupported items go to `do_not_claim` |


In [ ]:
# Five-step Analyze crew sequence

STEPS = [
    ("1 Resume reader", "Structured CandidateProfile"),
    ("2 JD reader", "JobRequirements"),
    ("3 Gap analyst", "Coverage score + GapReport"),
    ("4 Rewrite coach", "Cited resume rewrites"),
    ("5 Prep advisor", "InterviewPrepPlan (seniority-aware)"),
]

print("Analyze crew sequence")
for name, out in STEPS:
    print(f"  {name:20} -> {out}")

print()
print("Fallback:")
print("  crew fails / no key -> gap-based prep plan")
print("  still grounded in the resume and the job description")


## Guardrails

- **Input:** file type/size, one resume source, one JD source, no raw job-board scrape.
- **Tools:** upload confinement.
- **Output:** Pydantic validation; prep advice stays grounded in real gaps.

Implementation: `careercrew/guardrails.py`.


## Code reference

| Step | File | Symbol |
|------|------|--------|
| UI | `callbacks_analyze.py` | `analyze` |
| Orchestrator | `pipeline.py` | `run_analysis`, `build_fallback_plan` |
| Crew | `crew.py` | `build_crew` |
| MCP | `mcp_server/server.py` | two tools |
| HITL handoff | `review.py` | `submit_for_review` |

Continue with `04_hitl_and_coaches.ipynb`.
